# Results Class Overview

`search_data()` and `search_datasets()` return **lazy result containers** — `GranuleResults` and `CollectionResults`, both subclasses of `SearchResults` — instead of plain lists. Nothing is fetched from CMR until you ask for it.

This notebook showcases the container's behavior: lazy loading, the new `repr`, iteration, indexing/slicing, pagination, filtering, and fetching everything with a single call.

## Setup

Login so `earthaccess` can build an authenticated session for indirect (HTTPS) access.

In [ ]:
import earthaccess

auth = earthaccess.login()
auth

## 1. Search returns a lazy container

`search_data()` returns a `GranuleResults` object. The result type is shown in the `repr` — a small batch is fetched up front (`prefetch`, default 20), but the rest of the search is not downloaded until you request it.

In [ ]:
results = earthaccess.search_data(
    short_name="HLSL30",
    temporal=("2025-01-01", "2025-01-20"),
    bounding_box=(40.19, 6.24, 42.18, 7.24),
)
type(results)

### The new `repr`

The container's `repr` shows the **total** CMR hit count and how many results are currently **loaded** in memory:

    GranuleResults(total=1234, loaded=20)

In [ ]:
print(repr(results))

In a notebook, the `_repr_html_` renders as a collapsible table instead (just display the object).

In [ ]:
results

## 2. Total vs loaded

- `total()` — the number of matches reported by CMR (`CMR-Hits`).
- `len(results)` — the number of results **currently retained in memory**, not the total.

Iterating is a *stream*: `for granule in results:` yields every match but only
keeps a small prefetched window, so a huge search never piles up in memory.
Use `all()` to fetch **and cache** every result, or `pages()`/`items()` to
process incrementally without materializing.


In [ ]:
print("total matching in CMR:", results.total())
print("loaded so far (len):", len(results))
print("hits() is an alias for total():", results.hits() == results.total())

## 3. Indexing and slicing

Results support both integer indexing and slices. Any index past the prefetched batch is fetched on demand.

In [ ]:
first = results[0]
print("results[0] is a", type(first).__name__)
print("GranuleUR:", first["umm"].get("GranuleUR"))
print("concept-id:", first["meta"]["concept-id"])

In [ ]:
batch = results[0:5]
print(f"slice [0:5] -> {len(batch)} granules")
print(f"last granule via results[-1]: {results[-1]['umm'].get('GranuleUR')}")

## 4. Iteration

Iterating fetches additional pages lazily as needed. Use `items()` — the
explicit, pystac-client-compatible way to iterate one result at a time
(equivalent to iterating over the container directly). To inspect just the
first few lazily, wrap it in `itertools.islice`.


In [ ]:
from itertools import islice

for granule in islice(results.items(), 5):
    print(granule["umm"].get("GranuleUR"))

## 5. Fetching everything: `all()`

For a small search, `all()` fetches the remaining results and returns a plain list. Results are cached, so a second call is instant.

In [ ]:
granules = results.all()
print(f"all() returned {len(granules)} granules")
print(f"total() == len after all(): {results.total()} vs {len(granules)}")

## 6. Pagination: `pages()` and `items()`

For large result sets, process results incrementally instead of loading everything:

- `pages(page_size=...)` — iterate over pages (lists) of results.
- `items()` — iterate one result at a time (pystac-client compatible).

In [ ]:
for page_num, page in enumerate(results.pages(page_size=50)):
    print(f"page {page_num}: {len(page)} results")
    if page_num >= 1:
        break

In [ ]:
for granule in results.items():
    print("item:", granule["umm"].get("GranuleUR"))
    break

## 7. Filtering results

`filter()` fetches all results and applies criteria — either keyword shortcuts (`min_size`, `max_size`, `cloud_hosted`), a `GranuleFilter`, or a predicate function.

In [ ]:
from earthaccess.search import GranuleFilter

big = results.filter(min_size=50)
print(f"granules >= 50 MB: {len(big)}")

cloud = results.filter(cloud_hosted=True)
print(f"cloud-hosted granules: {len(cloud)}")

f = GranuleFilter(min_size=10, max_size=200)
filtered = results.filter(f)
print(f"granules between 10-200 MB: {len(filtered)}")

## 8. Summarize and map

- `summary()` — aggregate metadata (total, loaded, size, cloud count, temporal range).
- `explore()` — an interactive map of the results' spatial extents (needs the `[widgets]` extra).

In [ ]:
results.summary()

In [ ]:
# Requires: pip install "earthaccess[widgets]"
# results.explore()

## 9. Convert to STAC

`to_stac()` converts the loaded results to `pystac` objects — see the [CMR to STAC Semantics](../cmr-to-stac.ipynb) notebook for details.

In [ ]:
items = results.to_stac()
print(f"converted {len(items)} items")
print(items[0].id)

## 10. Collections are lazy too

`search_datasets()` returns a `CollectionResults` container with the same interface.

In [ ]:
datasets = earthaccess.search_datasets(keyword="sea surface temperature", count=10)
print(repr(datasets))
print("total:", datasets.total())
print("first:", datasets[0]["umm"].get("ShortName"))

## Summary

- `search_data()` / `search_datasets()` return **lazy** containers; `len()` is loaded-so-far, `total()` is the CMR hit count.
- The `repr` shows `total` and `loaded` at a glance; notebooks get a collapsible HTML table.
- Iteration is streaming — a large `for` keeps memory bounded; `all()`/`pages()`/`items()`/`filter()`/`summary()`/`to_stac()`/`explore()` cover the common workflows without pulling the whole set unless you want it.

See the [Results Class tutorials](index.md) and the [API reference](../../api/granules/granules.md) for the complete method list.